<a href="https://colab.research.google.com/github/ksehar99/EyePACS-DL-Blockchain/blob/main/DR_Private_Blockchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Blockchain Setup

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# NVM install + Node 22 + Python libraries
!wget -qO- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash > /dev/null 2>&1
!pip install web3 faker -q

print("Done!")

Done!


In [22]:
import os
os.makedirs("/content/hardhat-project", exist_ok=True)
os.chdir("/content/hardhat-project")

# Purane config files delete karo
!rm -f hardhat.config.cjs hardhat.config.js hardhat.config.ts

# package.json
with open("package.json", "w") as f:
    f.write('{"name":"dr-blockchain","version":"1.0.0"}')

# hardhat.config.js — clean, no comments inside object
with open("hardhat.config.js", "w") as f:
    f.write('module.exports = {\n  solidity: "0.8.22",\n  networks: { hardhat: { chainId: 1337 } }\n};\n')

# Confirm file content
!cat hardhat.config.js

# Node 22 + Hardhat install
!bash -c "source /root/.nvm/nvm.sh && nvm install 22 && nvm use 22 && npm install --save-dev hardhat@2.22.17 --legacy-peer-deps" 2>&1 | tail -3

!bash -c "source /root/.nvm/nvm.sh && nvm use 22 && ./node_modules/.bin/hardhat --version"

module.exports = {
  solidity: "0.8.22",
  networks: { hardhat: { chainId: 1337 } }
};
  npm audit fix --force

Run `npm audit` for details.
Now using node v22.23.1 (npm v10.9.8)
2.22.17


In [36]:
import subprocess, time, os
from web3 import Web3

os.chdir("/content/hardhat-project")

process = subprocess.Popen(
    ["bash", "-c", "source /root/.nvm/nvm.sh && nvm use 22 && ./node_modules/.bin/hardhat node"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("starting network...")
time.sleep(10)

w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))
if w3.is_connected():
    print(f"Private network started working! Chain ID: {w3.eth.chain_id}")
    print(f"Total accounts: {len(w3.eth.accounts)}")
    for i, acc in enumerate(w3.eth.accounts[:3]):
        bal = w3.from_wei(w3.eth.get_balance(acc), 'ether')
        print(f"  [{i}] {acc} — {bal} ETH")
else:
    _, err = process.communicate(timeout=2)
    print("Error:", err)

starting network...
Private network started working! Chain ID: 1337
Total accounts: 20
  [0] 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 — 9999.994875100011733788 ETH
  [1] 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 — 9999.99847170933871363 ETH
  [2] 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC — 10000 ETH


### ABI and Bytecode

In [53]:
import json

ABI = json.loads('''[
	{
		"inputs": [],
		"stateMutability": "nonpayable",
		"type": "constructor"
	},
	{
		"inputs": [],
		"name": "ConsentAlreadyGiven",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "ConsentNotGiven",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "NoDiagnosisFound",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "NotAuthorized",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "NotOwner",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "PatientAlreadyExists",
		"type": "error"
	},
	{
		"inputs": [],
		"name": "PatientNotFound",
		"type": "error"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "ConsentGiven",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "ConsentRevoked",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "doctorId",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "bool",
				"name": "diagnosisResult",
				"type": "bool"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "confidenceScore",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "DiagnosisUploaded",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "bool",
				"name": "doctorResult",
				"type": "bool"
			},
			{
				"indexed": false,
				"internalType": "bool",
				"name": "agreedWithAI",
				"type": "bool"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "DoctorDecisionRecorded",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "oldDoctor",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "newDoctor",
				"type": "address"
			}
		],
		"name": "DoctorReassigned",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "admin",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "reason",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "EmergencyAccessLog",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "doctorAddress",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "patientAddress",
				"type": "address"
			}
		],
		"name": "PatientRegistered",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"indexed": false,
				"internalType": "address",
				"name": "secondDoctor",
				"type": "address"
			},
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			}
		],
		"name": "SecondOpinionRequested",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "checkConsent",
		"outputs": [
			{
				"internalType": "bool",
				"name": "",
				"type": "bool"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "reason",
				"type": "string"
			}
		],
		"name": "emergencyAccess",
		"outputs": [
			{
				"components": [
					{
						"internalType": "bytes32",
						"name": "imageHash",
						"type": "bytes32"
					},
					{
						"internalType": "bool",
						"name": "diagnosisResult",
						"type": "bool"
					},
					{
						"internalType": "uint256",
						"name": "confidenceScore",
						"type": "uint256"
					},
					{
						"internalType": "bytes32",
						"name": "modelVersionHash",
						"type": "bytes32"
					},
					{
						"internalType": "uint256",
						"name": "patientId",
						"type": "uint256"
					},
					{
						"internalType": "address",
						"name": "doctorId",
						"type": "address"
					},
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "bool",
						"name": "reviewedByDoctor",
						"type": "bool"
					}
				],
				"internalType": "struct DRDiagnosisResults.Diagnosis[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "getPatientAddress",
		"outputs": [
			{
				"internalType": "address",
				"name": "",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "getPatientDoctor",
		"outputs": [
			{
				"internalType": "address",
				"name": "",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "giveConsent",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "isPatientRegistered",
		"outputs": [
			{
				"internalType": "bool",
				"name": "",
				"type": "bool"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "isReviewed",
		"outputs": [
			{
				"internalType": "bool",
				"name": "",
				"type": "bool"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			},
			{
				"internalType": "address",
				"name": "_newDoctor",
				"type": "address"
			}
		],
		"name": "reassignDoctor",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"internalType": "bool",
				"name": "doctorResult",
				"type": "bool"
			},
			{
				"internalType": "bool",
				"name": "agreedWithAI",
				"type": "bool"
			},
			{
				"internalType": "string",
				"name": "notes",
				"type": "string"
			}
		],
		"name": "recordDoctorDecision",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			},
			{
				"internalType": "address",
				"name": "_doctorAddress",
				"type": "address"
			},
			{
				"internalType": "address",
				"name": "_patientAddress",
				"type": "address"
			}
		],
		"name": "registerPatient",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"internalType": "address",
				"name": "secondDoctor",
				"type": "address"
			}
		],
		"name": "requestSecondOpinion",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			}
		],
		"name": "revokeConsent",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"internalType": "bytes32",
				"name": "imageHash",
				"type": "bytes32"
			},
			{
				"internalType": "bool",
				"name": "diagnosisResult",
				"type": "bool"
			},
			{
				"internalType": "uint256",
				"name": "confidenceScore",
				"type": "uint256"
			},
			{
				"internalType": "bytes32",
				"name": "modelVersionHash",
				"type": "bytes32"
			}
		],
		"name": "uploadDiagnosis",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "patientId",
				"type": "uint256"
			},
			{
				"internalType": "bytes32",
				"name": "imageHash",
				"type": "bytes32"
			}
		],
		"name": "verifyDiagnosis",
		"outputs": [
			{
				"internalType": "bool",
				"name": "",
				"type": "bool"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			}
		],
		"name": "viewDoctorDecisions",
		"outputs": [
			{
				"components": [
					{
						"internalType": "uint256",
						"name": "patientId",
						"type": "uint256"
					},
					{
						"internalType": "bool",
						"name": "doctorResult",
						"type": "bool"
					},
					{
						"internalType": "bool",
						"name": "agreedWithAI",
						"type": "bool"
					},
					{
						"internalType": "string",
						"name": "notes",
						"type": "string"
					},
					{
						"internalType": "address",
						"name": "doctorId",
						"type": "address"
					},
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					}
				],
				"internalType": "struct DRDiagnosisResults.DoctorDecision[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			}
		],
		"name": "viewMyRecords",
		"outputs": [
			{
				"components": [
					{
						"internalType": "bytes32",
						"name": "imageHash",
						"type": "bytes32"
					},
					{
						"internalType": "bool",
						"name": "diagnosisResult",
						"type": "bool"
					},
					{
						"internalType": "uint256",
						"name": "confidenceScore",
						"type": "uint256"
					},
					{
						"internalType": "bytes32",
						"name": "modelVersionHash",
						"type": "bytes32"
					},
					{
						"internalType": "uint256",
						"name": "patientId",
						"type": "uint256"
					},
					{
						"internalType": "address",
						"name": "doctorId",
						"type": "address"
					},
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "bool",
						"name": "reviewedByDoctor",
						"type": "bool"
					}
				],
				"internalType": "struct DRDiagnosisResults.Diagnosis[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "viewPatients",
		"outputs": [
			{
				"internalType": "uint256[]",
				"name": "",
				"type": "uint256[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "_patientId",
				"type": "uint256"
			}
		],
		"name": "viewRecords",
		"outputs": [
			{
				"components": [
					{
						"internalType": "bytes32",
						"name": "imageHash",
						"type": "bytes32"
					},
					{
						"internalType": "bool",
						"name": "diagnosisResult",
						"type": "bool"
					},
					{
						"internalType": "uint256",
						"name": "confidenceScore",
						"type": "uint256"
					},
					{
						"internalType": "bytes32",
						"name": "modelVersionHash",
						"type": "bytes32"
					},
					{
						"internalType": "uint256",
						"name": "patientId",
						"type": "uint256"
					},
					{
						"internalType": "address",
						"name": "doctorId",
						"type": "address"
					},
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "bool",
						"name": "reviewedByDoctor",
						"type": "bool"
					}
				],
				"internalType": "struct DRDiagnosisResults.Diagnosis[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	}
]''')

In [54]:
# ── Bytecode ────────────────────────────────────────────────────────────
BYTECODE = "..."

### Network Connection


In [55]:
from web3 import Web3

# ── Connect to private network ──────────────────────────────────────────
w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))

if not w3.is_connected():
    print("Network se connect nahi ho saka — pehle Cell 4 (network start) chalao")
else:
    print(f"Connected! Chain ID: {w3.eth.chain_id}")

    # Accounts assign karo
    ADMIN   = w3.eth.accounts[0]   # Hospital Admin
    DOCTOR1 = w3.eth.accounts[1]   # Doctor 1
    DOCTOR2 = w3.eth.accounts[2]   # Doctor 2
    PATIENT1 = w3.eth.accounts[3]  # Patient 1
    PATIENT2 = w3.eth.accounts[4]  # Patient 2

    print(f"\nAdmin:   {ADMIN}")
    print(f"Doctor1: {DOCTOR1}")
    print(f"Doctor2: {DOCTOR2}")
    print(f"Patient1: {PATIENT1}")
    print(f"Patient2: {PATIENT2}")

    # Contract deploy karo
    Contract = w3.eth.contract(abi=ABI, bytecode=BYTECODE)
    tx_hash = Contract.constructor().transact({'from': ADMIN})
    tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    CONTRACT_ADDRESS = tx_receipt.contractAddress
    print(f"\nContract deployed at: {CONTRACT_ADDRESS}")

    # Contract instance banao
    contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)
    print("Contract ready!")

Connected! Chain ID: 1337

Admin:   0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Doctor1: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Doctor2: 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC
Patient1: 0x90F79bf6EB2c4f870365E785982E1f101E93b906
Patient2: 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65

Contract deployed at: 0x2279B7A0a67DB372996a5FaB50D91eAA73d2eBe6
Contract ready!


### Testing

In [27]:
import hashlib

# ── Helpers ────────────────────────────────────────────────────────────
def to_bytes32(hex_str):
    """String hash ko bytes32 mein convert karo"""
    return bytes.fromhex(hex_str)

def sha256_file(path):
    """File ka SHA-256 hash nikalo"""
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

def sha256_string(s):
    """String ka SHA-256 hash nikalo"""
    return hashlib.sha256(s.encode()).hexdigest()

# ── Step 1: Patient register karo (Admin call karta hai) ───────────────
patient_id = 5

tx = contract.functions.registerPatient(
    patient_id,
    DOCTOR1,
    PATIENT1
).transact({'from': ADMIN})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Patient {patient_id} registered!")

# ── Step 2: Patient consent deta hai (Patient khud call karta hai) ─────
tx = contract.functions.giveConsent(
    patient_id
).transact({'from': PATIENT1})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Consent given by Patient {patient_id}!")

# ── Step 3: Consent verify karo ────────────────────────────────────────
consent = contract.functions.checkConsent(patient_id).call()
print(f"Consent status: {consent}")

Patient 5 registered!
Consent given by Patient 5!
Consent status: True


In [56]:
# # uncomment and run it when blockhcian restarted..
# import json, os
# path = "/content/drive/MyDrive/EyePACS-DL-Blockchain/used_accounts.json"
# with open(path, 'w') as f:
#     json.dump([], f)
# print("Used accounts reset!")

Used accounts reset!


### CLI setup


In [57]:
import json
import os
import hashlib
import random
from web3 import Web3
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
from web3.exceptions import ContractLogicError

# ── Setup and Configuration ───────────────────────
USED_ACCOUNTS_FILE = "/content/drive/MyDrive/EyePACS-DL-Blockchain/used_accounts.json"
USED_IMAGES_FILE   = "/content/drive/MyDrive/EyePACS-DL-Blockchain/used_images.json"
TEST_IMAGES_DIR    = "/content/drive/MyDrive/EyePACS/EyePACS/test/test_merge"
MODEL_VERSION      = "dr_detection_mobilenetv2_v2_threshold_0.30"

# Fixed role assignments for accounts
ADMIN_IDX   = 0
DOCTOR_IDXS = [1, 2, 3]
PATIENT_POOL = list(range(4, 20))

accounts = w3.eth.accounts
# Generate SHA-256 hash of the model version string
MODEL_VERSION_HASH = bytes.fromhex(
    hashlib.sha256(MODEL_VERSION.encode()).hexdigest()
)

# ── Helper Functions ───────────────────────────
def load_json(path, default):
    """Loads JSON data from a file, returns default if file not found or invalid."""
    try:
        with open(path) as f:
            return json.load(f)
    except:
        return default

def save_json(path, data):
    """Saves JSON data to a file, creating directories if they don't exist."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def get_next_patient_account():
    """Finds the next available patient account from the pool."""
    used = load_json(USED_ACCOUNTS_FILE, [])
    for idx in PATIENT_POOL:
        if idx not in used:
            return idx
    return None

def mark_account_used(idx):
    """Marks an account as used in the tracking file."""
    used = load_json(USED_ACCOUNTS_FILE, [])
    if idx not in used:
        used.append(idx)
    save_json(USED_ACCOUNTS_FILE, used)

def get_random_image():
    """Selects a random, unused test image for diagnosis."""
    used = load_json(USED_IMAGES_FILE, [])
    all_images = [
        f for f in os.listdir(TEST_IMAGES_DIR)
        if f.endswith('.jpg') and f not in used
    ]
    if not all_images:
        return None, None
    chosen = random.choice(all_images)
    full_path = os.path.join(TEST_IMAGES_DIR, chosen)
    used.append(chosen)
    save_json(USED_IMAGES_FILE, used)
    return full_path, chosen

def image_sha256(path):
    """Calculates the SHA-256 hash of an image file."""
    with open(path, 'rb') as f:
        return bytes.fromhex(hashlib.sha256(f.read()).hexdigest())

def clear():
    """Prints a separator for better menu readability."""
    print("\n" + "="*50 + "\n")

def pause(msg="Press Enter to continue..."):
    """Pauses execution and waits for user input."""
    input(f"\n{msg}")

def identify_account(idx):
    """Identifies the role of an account based on its index."""
    if idx == ADMIN_IDX:
        return "admin"
    elif idx in DOCTOR_IDXS:
        return "doctor"
    else:
        return "patient"

def _handle_contract_error(e):
    """Handles common Solidity contract errors with user-friendly messages."""
    error_str = str(e)
    # Extract the hex data if available
    hex_data = None
    if isinstance(e, ContractLogicError) and e.data:
        hex_data = e.data
        if isinstance(hex_data, dict) and 'data' in hex_data:
            hex_data = hex_data['data']

    if hex_data and "ea8e4eb5" in hex_data:
        print("Error: Consent has not been given for this patient. This might indicate that the contract function still requires explicit patient consent, or that the patient is not properly registered.")
    elif hex_data and "30cd7471" in hex_data:
        print("Error: Not Authorized. Your account does not have the necessary permissions for this action.")
    elif "PatientAlreadyExists" in error_str:
        print("Error: This Patient ID is already registered.")
    elif "PatientNotFound" in error_str:
        print("Error: Patient ID does not exist.")
    elif "ConsentAlreadyGiven" in error_str:
        print("Error: Consent has already been given for this patient.")
    elif "NoDiagnosisFound" in error_str:
        print("Error: No diagnosis records found for this patient.")
    elif "NotOwner" in error_str:
        print("Error: You are not the owner of this contract.")
    else:
        print(f"An unexpected error occurred: {e}")

# Load the AI model once globally
print("Loading the AI model...")
MODEL = tf.keras.models.load_model("/content/drive/MyDrive/EyePACS/best_model_final.keras")
THRESHOLD = 0.30
print("AI model ready!")

def run_model(image_path):
    """
    Runs the MobileNetV2 model for DR prediction.
    Applies Test Time Augmentation (TTA) by averaging predictions from 4 image versions.
    Returns: (diagnosis_result: bool, confidence: int)
    """
    # Load and preprocess image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    img = tf.expand_dims(img, 0)  # Add batch dimension

    # Test Time Augmentation (TTA) — average 4 versions
    preds = []
    preds.append(MODEL.predict(img, verbose=0)[0][0])
    preds.append(MODEL.predict(tf.image.flip_left_right(img), verbose=0)[0][0])
    preds.append(MODEL.predict(tf.image.flip_up_down(img), verbose=0)[0][0])
    preds.append(
        MODEL.predict(
            tf.image.flip_left_right(tf.image.flip_up_down(img)), verbose=0
        )[0][0]
    )

    avg_prob = float(np.mean(preds))
    diagnosis_result = avg_prob >= THRESHOLD
    confidence = int(avg_prob * 100)

    return diagnosis_result, confidence

# ─────────────────────────────────────────────
# ADMIN MENU
# ─────────────────────────────────────────────
def admin_menu():
    """Provides administrative functionalities like patient registration, doctor reassignment, and emergency access."""
    while True:
        clear()
        print("ADMIN PANEL")
        print("-" * 30)
        print("[1] Register new patient")
        print("[2] Reassign doctor")
        print("[3] View all patients")
        print("[4] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            # Get the next available account for a new patient
            acc_idx = get_next_patient_account()
            if acc_idx is None:
                print("No available patient accounts left in the pool.")
                pause()
                continue

            patient_address = accounts[acc_idx]

            # Automatically assign Patient ID
            # Retrieve all PatientRegistered events
            events = contract.events.PatientRegistered.get_logs(from_block=0, to_block='latest')
            if events:
                # Find the maximum patientId from existing events and increment by 1
                max_patient_id = max(e['args']['patientId'] for e in events)
                patient_id = max_patient_id + 1
            else:
                # If no patients registered yet, start with Patient ID 1
                patient_id = 1

            # Doctor selection for the new patient
            print("\nAvailable doctors:")
            for i, d in enumerate(DOCTOR_IDXS):
                print(f"  [{i+1}] Doctor {i+1}")
            try:
                d_choice = int(input("Select a doctor for this patient (1/2/3): ").strip()) - 1
                doctor_address = accounts[DOCTOR_IDXS[d_choice]]
            except (ValueError, IndexError):
                print("Invalid doctor selection. Please choose 1, 2, or 3.")
                pause()
                continue

            # Register patient on-chain
            try:
                tx = contract.functions.registerPatient(
                    patient_id,
                    doctor_address,
                    patient_address
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)
                mark_account_used(acc_idx)

                print(f"\nPatient successfully registered!")
                print(f"Patient ID : {patient_id}")
                print(f"Account    : [{acc_idx}] {patient_address}")
                print(f"Doctor     : Doctor {d_choice+1}")
                print(f"\nInform the patient: Log in using account number {acc_idx}")
            except Exception as e:
                print("Error during patient registration:")
                _handle_contract_error(e)
            pause()

        elif choice == "2":
            try:
                patient_id = int(input("Enter Patient ID: ").strip())

                # Check if patient is registered using getPatientDoctor function
                try:
                    _ = contract.functions.getPatientDoctor(patient_id).call()
                except Exception as e:
                    error_str = str(e)
                    if "PatientNotFound" in error_str:
                        print(f"Error: Patient {patient_id} not registered")
                        input("Press Enter...")
                        continue
                    else:
                        _handle_contract_error(e)
                        input("Press Enter...")
                        continue

                print("\nAvailable doctors:")
                for i, d in enumerate(DOCTOR_IDXS):
                    print(f"  [{i+1}] Doctor {i+1}")
                try:
                    d_choice = int(input("Select the new doctor (1/2/3): ").strip())
                    if d_choice not in [1, 2, 3]:
                        print("Invalid selection — 1, 2 ya 3 enter karo.")
                        input("Press Enter...")
                        continue
                    new_doctor = accounts[DOCTOR_IDXS[d_choice - 1]]
                except:
                    print("Invalid input.")
                    input("Press Enter...")
                    continue

                tx = contract.functions.reassignDoctor(
                    patient_id, new_doctor
                ).transact({'from': accounts[ADMIN_IDX]})
                w3.eth.wait_for_transaction_receipt(tx)
                print(f"Doctor successfully reassigned for Patient ID {patient_id}!")
            except Exception as e:
                print("Error during doctor reassignment:")
                _handle_contract_error(e)
            pause()

        elif choice == "3":
          print("\nAll registered patients:\n")
          events = contract.events.PatientRegistered.get_logs(from_block=0)
          if not events:
              print("Koi patient registered nahi hai abhi.")
          else:
              for e in events:
                  pid = e['args']['patientId']
                  # Changed: Get live doctor address from contract state
                  doctor = contract.functions.getPatientDoctor(pid).call()
                  doctor_num = "Unknown"
                  for i, d in enumerate(DOCTOR_IDXS):
                      if accounts[d].lower() == doctor.lower():
                          doctor_num = f"Doctor {i+1}"
                          break
                  print(f"  Patient ID: {pid} | Assigned to: {doctor_num}")
          input("\nPress Enter to continue...")

        elif choice == "4": # This was previously '5' for exit
            break

# ─────────────────────────────────────────────
# DOCTOR MENU
# ─────────────────────────────────────────────
def doctor_menu(doctor_idx):
    """Provides functionalities for doctors to manage patients, review diagnoses, and upload new diagnoses."""
    doctor_address = accounts[doctor_idx]
    doctor_num = DOCTOR_IDXS.index(doctor_idx) + 1

    while True:
        clear()
        print(f"DOCTOR {doctor_num} PANEL")
        print(f"Address: {doctor_address}")
        print("-" * 30)
        print("[1] View my assigned patients")
        print("[2] Upload new diagnosis and record decision")
        print("[3] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            try:
                patients = contract.functions.viewPatients().call(
                    {'from': doctor_address}
                )
                print(f"\nAssigned patients: {patients if patients else 'None currently assigned.'}")
            except Exception as e:
                print("Error viewing assigned patients:")
                _handle_contract_error(e)
            pause()

        elif choice == "2":
            try:
                patient_id = int(input("Enter Patient ID for diagnosis: ").strip())

                # Check 1: Patient exists?
                try:
                    # Attempt to call a patient-specific view function (e.g., checkConsent)
                    # This should revert with PatientNotFound if the ID is not registered.
                    # Using a dummy call to check existence, not actual consent status
                    contract.functions.checkConsent(patient_id).call({'from': doctor_address})
                except Exception as e:
                    error_str = str(e)
                    if "PatientNotFound" in error_str:
                        print(f"Error: Patient {patient_id} does not exist.")
                    else:
                        _handle_contract_error(e) # Handle other unexpected errors during existence check
                    pause()
                    continue # Skip to next loop iteration

                # Check 2: Is this my patient?
                my_patients = contract.functions.viewPatients().call({'from': doctor_address})
                if patient_id not in my_patients:
                    print(f"Error: Patient {patient_id} is not assigned to you.")
                    pause()
                    continue # Skip to next loop iteration

                # Retrieve a random, unused image
                image_path, image_name = get_random_image()
                if image_path is None:
                    print("No unused test images available in the specified directory.")
                    pause()
                    continue

                print(f"\nImage selected for diagnosis: {image_name}")
                print("Running AI model prediction...")

                # Run AI model prediction
                diagnosis_result, confidence = run_model(image_path)

                print(f"\nAI Prediction Results:")
                print(f"  Result    : {'DR Detected' if diagnosis_result else 'No DR Detected'}")
                print(f"  Confidence: {confidence}%")

                # Doctor's decision based on AI prediction
                print("\nDoctor's Decision:")
                print("[a] Agree with AI prediction")
                print("[o] Override AI prediction")
                d_input = input("Select (a/o): ").strip().lower()

                agreed = d_input == 'a'
                if agreed:
                    doctor_result = diagnosis_result
                else:
                    print("[1] DR Detected (Your diagnosis)")
                    print("[2] No DR Detected (Your diagnosis)")
                    r = input("Enter your diagnosis (1 or 2): ").strip()
                    doctor_result = r == "1"

                notes = input("Enter any additional notes (optional, press Enter to skip): ").strip()

                # Calculate image hash
                image_hash = image_sha256(image_path)

                # Upload diagnosis details on-chain
                tx1 = contract.functions.uploadDiagnosis(
                    patient_id,
                    image_hash,
                    diagnosis_result,
                    confidence,
                    MODEL_VERSION_HASH
                ).transact({'from': doctor_address})
                w3.eth.wait_for_transaction_receipt(tx1)

                # Record doctor's decision on-chain
                tx2 = contract.functions.recordDoctorDecision(
                    patient_id,
                    doctor_result,
                    agreed,
                    notes
                ).transact({'from': doctor_address})
                w3.eth.wait_for_transaction_receipt(tx2)

                print(f"\nDiagnosis uploaded and decision successfully recorded on-chain!")
                print(f"Image hash: {image_hash.hex()[:20]}...")

                # Verify diagnosis integrity (tamper-proofing)
                verified = contract.functions.verifyDiagnosis(
                    patient_id, image_hash
                ).call()
                print(f"Tamper verification: {'PASSED' if verified else 'FAILED - Data Mismatch!'}")

            except Exception as e:
                print("Error during diagnosis upload or decision recording:")
                _handle_contract_error(e)
            pause()

        elif choice == "3":
            break

# ─────────────────────────────────────────────
# PATIENT MENU
# ─────────────────────────────────────────────
def patient_menu(patient_address, patient_id=None):
    """Provides functionalities for patients to view their records and manage consent."""

    # Agar main() function se patient_id nahi aayi (direct address login), to hi event scan karein
    if patient_id is None:
        try:
            all_patient_registered_events = contract.events.PatientRegistered.get_logs(from_block=0, to_block='latest')
            for event_entry in all_patient_registered_events:
                event_args = event_entry['args']
                if event_args.get('patientAddress') and event_args['patientAddress'].lower() == patient_address.lower():
                    patient_id = event_args['patientId']
                    break
        except Exception:
            pass

    # Agar phir bhi na mile, to smart contract ki mapping se direct tasdeeq karein
    if patient_id is None:
        print("Patient ID not found for this account. Please contact the administrator for registration.")
        pause()
        return

    while True:
        clear()
        print(f"PATIENT PANEL")
        print(f"Patient ID: {patient_id}")
        print(f"Address   : {patient_address}")
        print("-" * 30)
        print("[1] View my diagnosis records")
        print("[2] View doctor's decisions")
        print("[3] Exit")
        print()
        choice = input("Select option: ").strip()

        if choice == "1":
            try:
                records = contract.functions.viewMyRecords(
                    patient_id
                ).call({'from': patient_address})

                if not records:
                    print("\nNo diagnosis records found for you yet.")
                else:
                    print(f"\nTotal diagnosis records: {len(records)}")
                    for i, r in enumerate(records):
                        print(f"\n  Diagnosis {i+1}:")
                        print(f"    AI Result : {'DR Detected' if r[1] else 'No DR'}")
                        print(f"    Confidence: {r[2]}%")
                        print(f"    Date      : Block timestamp {r[6]}")
                        print(f"    Reviewed  : {'Yes' if r[7] else 'Pending doctor review'}")
            except Exception as e:
                print("Error viewing diagnosis records:")
                _handle_contract_error(e)
            pause()

        elif choice == "2":
            try:
                decisions = contract.functions.viewDoctorDecisions(
                    patient_id
                ).call({'from': patient_address})

                if not decisions:
                    print("\nNo doctor decisions found for you yet.")
                else:
                    for i, d in enumerate(decisions):
                        print(f"\n  Decision {i+1}: ")
                        print(f"    Doctor's Result : {'DR Detected' if d[1] else 'No DR'}")
                        print(f"    Agreed with AI: {'Yes' if d[2] else 'No (Overridden by Doctor)'}")
                        print(f"    Notes           : {d[3] if d[3] else 'None provided'}")
            except Exception as e:
                print("Error viewing doctor's decisions:")
                _handle_contract_error(e)
            pause()

        elif choice == "3":
            break

# ─────────────────────────────────────────────
# MAIN APPLICATION LOGIN
# ─────────────────────────────────────────────
def main():
    """Main function to handle user login and route to respective role-based menus."""
    while True:
        clear()
        print("╔══════════════════════════════════════════════╗")
        print("║        DR DIAGNOSIS BLOCKCHAIN SYSTEM        ║")
        print("║        Private Network  |  Chain ID 1337     ║")
        print("╚══════════════════════════════════════════════╝")
        print("\nSelect your role by entering the corresponding ID:")
        print("  [0]       - Hospital Admin")
        print("  [1, 2, 3] - Doctors")
        print("  [P]       - Patient Login (Using Patient ID)")
        print("  [99]      - Exit system")
        print()

        raw_input = input("Enter your ID or Option: ").strip()

        # Exit condition
        if raw_input == "99":
            print("\nExiting system...")
            break

        # ── 1. PATIENT LOGIN HANDLING (Using Patient ID) ──
        if raw_input.upper() == "P":
            pid_input = input("Enter your Patient ID: ").strip()
            try:
                patient_id = int(pid_input)
                address = None

                print("\nBlockchain storage se mapping verify ki ja rahi hai...")

                # PatientRegistered events se patientId -> patientAddress dhoondo
                try:
                    events = contract.events.PatientRegistered.get_logs(from_block=0, to_block='latest')
                    for e in events:
                        if e['args']['patientId'] == patient_id:
                            address = e['args']['patientAddress']
                            break
                except Exception:
                    pass

                if address:
                    print(f"\nPatient ID {patient_id} login confirmed!")
                    print(f"Connected Address: {address}")
                    pause()
                    patient_menu(address, patient_id)
                else:
                    print(f"Error: Patient ID {patient_id} not found or not registered.")
                    pause()

            except ValueError:
                print("Invalid Patient ID. Please enter a number.")
                pause()
            continue # Go back to main menu selection

        # ── 2. ADMIN & DOCTOR LOGIN HANDLING (Using Account Index) ──
        try:
            idx = int(raw_input)
            if not (0 <= idx <= 19):
                raise ValueError
        except ValueError:
            print("Invalid input — please enter 0, 1, 2, 3, P (or 99 to exit).")
            pause()
            continue

        address = accounts[idx]
        role = identify_account(idx)

        # Admin Login
        if role == "admin":
            print(f"\nAdmin login confirmed for account [{idx}].")
            print(f"Address: {address}")
            pause()
            admin_menu()

        # Doctor Login
        elif role == "doctor":
            doctor_num = DOCTOR_IDXS.index(idx) + 1
            print(f"\nDoctor {doctor_num} login confirmed for account [{idx}].")
            print(f"Address: {address}")
            pause()
            doctor_menu(idx)

        # Generic index login for patients is blocked to avoid confusion
        elif role == "patient":
            print(f"\nDirect account indexing for patients is disabled.")
            print("Please use the '[P]' option to login using your unique Patient ID.")
            pause()

# ── Run Main Application ────────────────────────────
main()


Loading the AI model...
AI model ready!


╔══════════════════════════════════════════════╗
║        DR DIAGNOSIS BLOCKCHAIN SYSTEM        ║
║        Private Network  |  Chain ID 1337     ║
╚══════════════════════════════════════════════╝

Select your role by entering the corresponding ID:
  [0]       - Hospital Admin
  [1, 2, 3] - Doctors
  [P]       - Patient Login (Using Patient ID)
  [99]      - Exit system

Enter your ID or Option: 0

Admin login confirmed for account [0].
Address: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266

Press Enter to continue...


ADMIN PANEL
------------------------------
[1] Register new patient
[2] Reassign doctor
[3] View all patients
[4] Exit

Select option: 1

Available doctors:
  [1] Doctor 1
  [2] Doctor 2
  [3] Doctor 3
Select a doctor for this patient (1/2/3): 1

Patient successfully registered!
Patient ID : 1
Account    : [4] 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65
Doctor     : Doctor 1

Inform the patient: Log in using account number 4

Press E

In [49]:
# Confirm karo ke naya contract hai
print("Contract address:", CONTRACT_ADDRESS)

# viewDoctorDecisions ka test
# Patient 1 ki address
pat_addr = None
try:
    all_patient_registered_events = contract.events.PatientRegistered.get_logs(from_block=0, to_block='latest')
    for event_entry in all_patient_registered_events:
        event_args = event_entry['args']
        if event_args.get('patientId') == 1:
            pat_addr = event_args['patientAddress']
            break
except Exception as e:
    print(f"Error fetching patient address from events: {e}")

if pat_addr:
    print("Patient 1 address:", pat_addr)
    # Direct call karo
    try:
        decisions = contract.functions.viewDoctorDecisions(1).call({'from': pat_addr})
        print("Decisions:", decisions)
    except Exception as e:
        print("Error:", e)
else:
    print("Patient ID 1's address not found in registered events.")

Contract address: 0x0165878A594ca255338adfa4d48449f69242Eb8F
Patient 1 address: 0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc
Error: ('execution reverted: Error: VM Exception while processing transaction: reverted with an unrecognized custom error (return data: 0xea8e4eb5)', {'message': 'Error: VM Exception while processing transaction: reverted with an unrecognized custom error (return data: 0xea8e4eb5)', 'data': '0xea8e4eb5'})


In [52]:
for item in ABI:
    if item.get('type') == 'function':
        print(item.get('name'))

checkConsent
emergencyAccess
getPatientDoctor
giveConsent
isReviewed
reassignDoctor
recordDoctorDecision
registerPatient
requestSecondOpinion
revokeConsent
uploadDiagnosis
verifyDiagnosis
viewDoctorDecisions
viewMyRecords
viewPatients
viewRecords


In [30]:
import os

def get_image_label(image_name):
    """
    EyePACS filename se label extract karo.
    X_ prefix = No DR (class 0)
    Y_ prefix = DR (class 1,2,3,4)
    """
    if image_name.startswith('X_'):
        return 0, "No DR"
    elif image_name.startswith('Y_'):
        return 1, "DR"
    else:
        return -1, "Unknown"

# Last used image check karo
import json
used = json.load(open("/content/drive/MyDrive/EyePACS-DL-Blockchain/used_images.json"))

if not used:
    print("Koi image abhi tak use nahi hui.")
else:
    print(f"{'Image':<45} {'True Label':<12} {'Prefix'}")
    print("-" * 65)
    for img in used:
        label_num, label_str = get_image_label(img)
        prefix = img[0] if img else "?"
        print(f"{img:<45} {label_str:<12} {prefix}")

Image                                         True Label   Prefix
-----------------------------------------------------------------
X_eyepacs_0_2436_right..jpg                   No DR        X
X_eyepacs_0_2401_right..jpg                   No DR        X
X_eyepacs_0_2503_left..jpg                    No DR        X
X_eyepacs_1_27304_left..jpg                   No DR        X
X_eyepacs_0_2500_left..jpg                    No DR        X


what i need to do is
- testing of patient login and patient features..
- final complete testing
-
